<a href="https://colab.research.google.com/github/MariamHazem226/AI-in-Software-Debugging-Research/blob/main/HumanEval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq datasets -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
COMPLETE HUMANEVAL EVALUATION (n=30)
Methods 1-12 using shared agents
"""

import re
import time
import os
import json
from groq import Groq
from datasets import load_dataset

MODEL = "llama-3.3-70b-versatile"
# Replace with your Groq API key
API_KEY = "your_actual_key_here"
client = Groq(api_key=API_KEY)

TOTAL = 30
RESULTS_PATH = "/content/drive/MyDrive/results_humaneval.json"

dataset = load_dataset("openai_humaneval", split="test")

print(f"HumanEval loaded: {len(dataset)} samples, evaluating first {TOTAL}")
print(f"Model: {MODEL}")
print("=" * 60)


def extract_code(text):
    patterns = [
        r"```python\s*(.*?)```",
        r"```\s*(.*?)```",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1).strip()
    return text.strip()


def safe_api_call(messages, temperature=0.2, max_tokens=400):
    wait = 60
    while True:
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e):
                match = re.search(r'try again in (\d+)m([\d.]+)s', str(e))
                if match:
                    wait = int(match.group(1)) * 60 + int(float(match.group(2))) + 5
                print(f"Rate limit hit, waiting {wait} seconds...")
                time.sleep(wait)
                wait = min(wait * 2, 300)
            else:
                raise


def make_test_code(sample):
    return f"{sample['test']}\ncheck({sample['entry_point']})"


def run_code(code, test_code):
    local_env = {}
    exec(code, local_env)
    exec(test_code, local_env)
    return True


def save_result(key, value):
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            all_results = json.load(f)
    else:
        all_results = {}
    all_results[key] = value
    with open(RESULTS_PATH, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Saved: {key} = {value:.1f}%")
    print(f"All results: {all_results}")


def evaluate_method(solve_func, method_name, delay=20):
    print(f"\n{'='*60}")
    print(f"EVALUATING: {method_name}")
    print(f"{'='*60}")
    passed = 0
    for idx, sample in enumerate(dataset.select(range(TOTAL))):
        print(f"\n[{idx+1}/{TOTAL}] {sample['entry_point']}")
        try:
            result = solve_func(sample["prompt"], make_test_code(sample))
            if result:
                passed += 1
                print(f"PASS ({passed}/{idx+1})")
            else:
                print(f"FAIL ({passed}/{idx+1})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(delay)
    accuracy = (passed / TOTAL) * 100
    print(f"\n{method_name} Accuracy: {passed}/{TOTAL} = {accuracy:.1f}%")
    return accuracy

print("Setup complete.")

HumanEval loaded: 164 samples, evaluating first 30
Model: llama-3.3-70b-versatile
Setup complete.


In [ ]:
# SHARED AGENTS - used across all methods
# Standard Agents
def planning_agent(problem):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a planning agent.

Problem:
{problem}

Return ONLY a numbered step-by-step plan.
No code. No explanation.
"""
        }],
        temperature=0.3,
        max_tokens=300
    )


def coding_agent(problem, plan):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a coding agent.

Problem:
{problem}

Plan:
{plan}

Return ONLY Python function implementation.
No explanation. No markdown.
"""
        }],
        temperature=0.2,
        max_tokens=400
    )


def debugging_agent(problem, code, error):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a debugging agent.

Fix the code.

Problem:
{problem}

Code:
{code}

Error:
{error}

Return ONLY corrected Python function.
No explanation.
"""
        }],
        temperature=0.2,
        max_tokens=400
    )


# CODESIM Agents
def codesim_planning_agent(problem):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a simulation planning agent.

Problem:
{problem}

First, simulate step-by-step what the function should do with example inputs.
Then write a numbered plan.

Return ONLY the simulation trace and plan.
No code.
"""
        }],
        temperature=0.3,
        max_tokens=500
    )


def codesim_coding_agent(problem, plan):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a coding agent.

Problem:
{problem}

Simulation and Plan:
{plan}

Return ONLY the Python function implementation.
No explanation. No markdown.
"""
        }],
        temperature=0.2,
        max_tokens=400
    )


def codesim_debugging_agent(problem, code, error):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a debugging agent.

Problem:
{problem}

Code:
{code}

Error:
{error}

Simulate the correct behavior first, then return ONLY the corrected Python function.
"""
        }],
        temperature=0.2,
        max_tokens=500
    )

print("Shared agents ready.")

Shared agents ready.


In [ ]:
# Standard Multi-Agent
def standard_ma_solve(problem, test_code, retries=2):
    try:
        plan = planning_agent(problem)
        code = extract_code(coding_agent(problem, plan))
        if not code:
            return False
        for attempt in range(retries + 1):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(debugging_agent(problem, code, str(e)))
                    if not code:
                        return False
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_standard_ma = evaluate_method(standard_ma_solve, "Standard Multi-Agent", delay=20)
save_result("standard_ma", acc_standard_ma)


EVALUATING: Standard Multi-Agent

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): name 'is_palindrome' is not defined
FAIL (attempt 2): 
FAIL (attempt 3): name 'is_palindrome' is not defined
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
PASSED (attempt 1)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
PASSED (attempt 1)
PASS (14/15)

[16/30] string_sequence
PASSE

In [ ]:
# Standard Self-Debug
def standard_sd_solve(problem, test_code, iterations=3):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Solve this problem.

Problem:
{problem}

Return ONLY the Python function.
No explanation. No markdown.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        for i in range(iterations):
            try:
                run_code(code, test_code)
                print(f"PASSED (iteration {i + 1})")
                return True
            except Exception as e:
                print(f"FAIL (iteration {i + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
The following code failed.

Problem:
{problem}

Code:
{code}

Error:
{str(e)}

Fix the code. Return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=400
                ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_standard_sd = evaluate_method(standard_sd_solve, "Standard Self-Debug", delay=20)
save_result("standard_sd", acc_standard_sd)


EVALUATING: Standard Self-Debug

[1/30] has_close_elements
PASSED (iteration 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (iteration 1)
PASS (2/2)

[3/30] truncate_number
PASSED (iteration 1)
PASS (3/3)

[4/30] below_zero
PASSED (iteration 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (iteration 1)
PASS (5/5)

[6/30] intersperse
PASSED (iteration 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (iteration 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (iteration 1)
PASS (8/8)

[9/30] sum_product
PASSED (iteration 1)
PASS (9/9)

[10/30] rolling_max
PASSED (iteration 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (iteration 1): name 'is_palindrome' is not defined
FAIL (iteration 2): 
PASSED (iteration 3)
PASS (11/11)

[12/30] string_xor
PASSED (iteration 1)
PASS (12/12)

[13/30] longest
PASSED (iteration 1)
PASS (13/13)

[14/30] greatest_common_divisor
PASSED (iteration 1)
PASS (14/14)

[15/30] all_prefixes
PASSED (iteration 1)
PASS (15/15)

[16/30] string_sequence
PASSED 

In [ ]:
# Standard Hybrid
def standard_hybrid_solve(problem, test_code, retries=3):
    try:
        plan = planning_agent(problem)
        code = extract_code(coding_agent(problem, plan))
        if not code:
            return False
        for attempt in range(retries):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt == 0:
                    code = extract_code(debugging_agent(problem, code, str(e)))
                else:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
Fix the following code.

Error:
{str(e)}

Code:
{code}

Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.1,
                        max_tokens=400
                    ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_standard_hybrid = evaluate_method(standard_hybrid_solve, "Standard Hybrid", delay=20)
save_result("standard_hybrid", acc_standard_hybrid)


EVALUATING: Standard Hybrid

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): name 'is_palindrome' is not defined
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
PASSED (attempt 1)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
PASSED (attempt 1)
PASS (14/15)

[16/30] string_sequence
PASSED (attempt 1)
PASS (15/16)

[17/30] coun

In [ ]:
# CODESIM Multi-Agent
def codesim_ma_solve(problem, test_code, retries=2):
    try:
        plan = codesim_planning_agent(problem)
        code = extract_code(codesim_coding_agent(problem, plan))
        if not code:
            return False
        for attempt in range(retries + 1):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(codesim_debugging_agent(problem, code, str(e)))
                    if not code:
                        return False
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_codesim_ma = evaluate_method(codesim_ma_solve, "CODESIM Multi-Agent", delay=20)
save_result("codesim_ma", acc_codesim_ma)


EVALUATING: CODESIM Multi-Agent

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
PASSED (attempt 1)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
PASSED (attempt 1)
PASS (14/15)

[16/30] string_sequence
PASSED (attempt 1)
PASS (15/16)

[17/30] count_distinct_characters
PASSED (a

In [ ]:
# CODESIM Self-Debug
def codesim_sd_solve(problem, test_code, iterations=3):
    try:
        response = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Simulate the correct behavior step-by-step for the following problem.
Show the simulation trace, then write the Python function.

Problem:
{problem}

Return the simulation trace AND the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=600
        )
        code = extract_code(response)
        for i in range(iterations):
            try:
                run_code(code, test_code)
                print(f"PASSED (iteration {i + 1})")
                return True
            except Exception as e:
                print(f"FAIL (iteration {i + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
Problem:
{problem}

Code:
{code}

Error:
{str(e)}

Simulate what went wrong, then return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=500
                ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_codesim_sd = evaluate_method(codesim_sd_solve, "CODESIM Self-Debug", delay=20)
save_result("codesim_sd", acc_codesim_sd)


EVALUATING: CODESIM Self-Debug

[1/30] has_close_elements
FAIL (iteration 1): invalid syntax (<string>, line 1)
PASSED (iteration 2)
PASS (1/1)

[2/30] separate_paren_groups
['()', '(())', '(()())']
PASSED (iteration 1)
PASS (2/2)

[3/30] truncate_number
PASSED (iteration 1)
PASS (3/3)

[4/30] below_zero
PASSED (iteration 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (iteration 1)
PASS (5/5)

[6/30] intersperse
[]
[1, 4, 2, 4, 3]
PASSED (iteration 1)
PASS (6/6)

[7/30] parse_nested_parens
FAIL (iteration 1): unterminated string literal (detected at line 3) (<string>, 
PASSED (iteration 2)
PASS (7/7)

[8/30] filter_by_substring
PASSED (iteration 1)
PASS (8/8)

[9/30] sum_product
(10, 24)
(0, 1)
PASSED (iteration 1)
PASS (9/9)

[10/30] rolling_max
[1, 2, 3, 3, 3, 4, 4]
PASSED (iteration 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (iteration 1): cannot access local variable 'postfix' where it is not assoc
FAIL (iteration 2): 
FAIL (iteration 3): 
FAIL (10/11)

[12/30] string_xor

In [ ]:
# CODESIM Hybrid
def codesim_hybrid_solve(problem, test_code, retries=3):
    try:
        plan = codesim_planning_agent(problem)
        code = extract_code(codesim_coding_agent(problem, plan))
        if not code:
            return False
        for attempt in range(retries):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt == 0:
                    code = extract_code(codesim_debugging_agent(problem, code, str(e)))
                else:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
Fix the following code.

Error:
{str(e)}

Code:
{code}

Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.1,
                        max_tokens=400
                    ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_codesim_hybrid = evaluate_method(codesim_hybrid_solve, "CODESIM Hybrid", delay=20)
save_result("codesim_hybrid", acc_codesim_hybrid)


EVALUATING: CODESIM Hybrid

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): name 'is_palindrome' is not defined
FAIL (attempt 2): 
FAIL (attempt 3): name 'is_palindrome' is not defined
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
PASSED (attempt 1)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
PASSED (attempt 1)
PASS (14/15)

[16/30] string_sequence
PASSED (att

In [ ]:
# Normal AgentCoder
def normal_agentcoder_solve(problem, test_code, retries=2):
    try:
        plan = planning_agent(problem)
        code = extract_code(coding_agent(problem, plan))
        if not code:
            return False
        safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Generate 2 edge-case assert statements for the following problem.

Problem:
{problem}

Return ONLY the assert statements, one per line.
"""
            }],
            temperature=0.3,
            max_tokens=150
        )
        for attempt in range(retries + 1):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(debugging_agent(problem, code, str(e)))
                    if not code:
                        return False
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_normal_agentcoder = evaluate_method(normal_agentcoder_solve, "Normal AgentCoder", delay=20)
save_result("normal_agentcoder", acc_normal_agentcoder)


EVALUATING: Normal AgentCoder

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): name 'is_palindrome' is not defined
FAIL (attempt 2): 
FAIL (attempt 3): name 'is_palindrome' is not defined
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
PASSED (attempt 1)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
PASSED (attempt 1)
PASS (14/15)

[16/30] string_sequence
PASSED (

In [ ]:
# Normal Codex Baseline
def normal_codex_solve(problem, test_code):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a Python function to solve the following problem.

Problem:
{problem}

Return ONLY the Python function.
No explanation. No markdown.
"""
            }],
            temperature=0.0,
            max_tokens=400
        ))
        run_code(code, test_code)
        print("PASSED")
        return True
    except Exception as e:
        print(f"FAIL: {str(e)[:60]}")
        return False


acc_normal_codex = evaluate_method(normal_codex_solve, "Normal Codex Baseline", delay=10)
save_result("normal_codex", acc_normal_codex)


EVALUATING: Normal Codex Baseline

[1/30] has_close_elements
PASSED
PASS (1/1)

[2/30] separate_paren_groups
PASSED
PASS (2/2)

[3/30] truncate_number
PASSED
PASS (3/3)

[4/30] below_zero
PASSED
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED
PASS (5/5)

[6/30] intersperse
PASSED
PASS (6/6)

[7/30] parse_nested_parens
PASSED
PASS (7/7)

[8/30] filter_by_substring
PASSED
PASS (8/8)

[9/30] sum_product
PASSED
PASS (9/9)

[10/30] rolling_max
PASSED
PASS (10/10)

[11/30] make_palindrome
FAIL: name 'is_palindrome' is not defined
FAIL (10/11)

[12/30] string_xor
PASSED
PASS (11/12)

[13/30] longest
FAIL: name 'Optional' is not defined
FAIL (11/13)

[14/30] greatest_common_divisor
PASSED
PASS (12/14)

[15/30] all_prefixes
FAIL: name 'List' is not defined
FAIL (12/15)

[16/30] string_sequence
PASSED
PASS (13/16)

[17/30] count_distinct_characters
PASSED
PASS (14/17)

[18/30] parse_music
PASSED
PASS (15/18)

[19/30] how_many_times
PASSED
PASS (16/19)

[20/30] sort_numbers
PASSED
PASS (17/20)

In [ ]:
# Normal MGDebugger
def normal_mgdebugger_solve(problem, test_code, retries=2):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a correct Python function for the following problem.

Problem:
{problem}

Return ONLY the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        decomp = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Break the following problem into small logical steps.

Problem:
{problem}

Return ONLY the steps. No code.
"""
            }],
            temperature=0.3,
            max_tokens=200
        )
        for attempt in range(retries + 1):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
You are a hierarchical debugger.

Problem steps:
{decomp}

Code:
{code}

Error:
{str(e)}

Fix the minimal part that caused the error.
Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.2,
                        max_tokens=500
                    ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_normal_mgdebugger = evaluate_method(normal_mgdebugger_solve, "Normal MGDebugger", delay=20)
save_result("normal_mgdebugger", acc_normal_mgdebugger)


EVALUATING: Normal MGDebugger

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
FAIL (attempt 1): name 'List' is not defined
PASSED (attempt 2)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
FAIL (attempt 1): name 'List' is not defined
PASSED (attempt 2)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
FAIL (attempt 1): name 'List' is not defined


In [ ]:
# AgentCoder Reference [19]
def ref_agentcoder_solve(problem, test_code, retries=3):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
You are a programmer agent.
Write a complete and correct Python function.

Problem:
{problem}

Return ONLY the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        extra_tests_raw = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
You are a test designer agent.
Generate 3 diverse assert statements to test the following function.

Problem:
{problem}

Return ONLY assert statements, one per line.
"""
            }],
            temperature=0.3,
            max_tokens=200
        )
        extra_tests = [l.strip() for l in extra_tests_raw.splitlines() if l.strip().startswith("assert")]
        for attempt in range(retries):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
You are a programmer agent.
Fix the following code based on the error.

Problem:
{problem}

Code:
{code}

Error:
{str(e)}

Return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=400
                ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_ref_agentcoder = evaluate_method(ref_agentcoder_solve, "AgentCoder [19]", delay=25)
save_result("ref_agentcoder", acc_ref_agentcoder)


EVALUATING: AgentCoder [19]

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
PASSED (attempt 1)
PASS (11/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (12/12)

[13/30] longest
FAIL (attempt 1): name 'List' is not defined
PASSED (attempt 2)
PASS (13/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (14/14)

[15/30] all_prefixes
PASSED (attempt 1)
PASS (15/15)

[16/30] string_sequence
PASSED (attempt 1)
PASS (16/16)

[17/30] count_distinct_characters
PASSED

In [ ]:
# Codex Baseline Reference [9]
def ref_codex_solve(problem, test_code):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": problem
            }],
            temperature=0.0,
            max_tokens=400
        ))
        run_code(code, test_code)
        print("PASSED")
        return True
    except Exception as e:
        print(f"FAIL: {str(e)[:60]}")
        return False


acc_ref_codex = evaluate_method(ref_codex_solve, "Codex Baseline [9]", delay=10)
save_result("ref_codex", acc_ref_codex)


EVALUATING: Codex Baseline [9]

[1/30] has_close_elements
False
True
PASSED
PASS (1/1)

[2/30] separate_paren_groups
['()', '(())', '(()())']
PASSED
PASS (2/2)

[3/30] truncate_number
PASSED
PASS (3/3)

[4/30] below_zero
False
True
PASSED
PASS (4/4)

[5/30] mean_absolute_deviation
1.0
PASSED
PASS (5/5)

[6/30] intersperse
[]
[1, 4, 2, 4, 3]
PASSED
PASS (6/6)

[7/30] parse_nested_parens
[2, 3, 1, 3]
PASSED
PASS (7/7)

[8/30] filter_by_substring
[]
['abc', 'bacd', 'array']
PASSED
PASS (8/8)

[9/30] sum_product
(0, 1)
(10, 24)
PASSED
PASS (9/9)

[10/30] rolling_max
[1, 2, 3, 3, 3, 4, 4]
PASSED
PASS (10/10)

[11/30] make_palindrome

catac
catac
PASSED
PASS (11/11)

[12/30] string_xor
100
PASSED
PASS (12/12)

[13/30] longest
None
a
ccc
PASSED
PASS (13/13)

[14/30] greatest_common_divisor
PASSED
PASS (14/14)

[15/30] all_prefixes
['a', 'ab', 'abc']
PASSED
PASS (15/15)

[16/30] string_sequence
PASSED
PASS (16/16)

[17/30] count_distinct_characters
PASSED
PASS (17/17)

[18/30] parse_music
[4,

In [ ]:
#  MGDebugger Reference [6]
def ref_mgdebugger_solve(problem, test_code, retries=3):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a correct Python function for the following problem.

Problem:
{problem}

Return ONLY the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        decomp = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Decompose the following problem into small sub-functions.
For each sub-function, describe what it does.

Problem:
{problem}

Return ONLY the decomposition. No code.
"""
            }],
            temperature=0.3,
            max_tokens=300
        )
        for attempt in range(retries):
            try:
                run_code(code, test_code)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
You are a hierarchical debugger.
Use the problem decomposition to identify and fix the bug.

Decomposition:
{decomp}

Code:
{code}

Error:
{str(e)}

Return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=500
                ))
                decomp = safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
Update the decomposition based on the failure.

Problem:
{problem}

Error:
{str(e)}

Return updated decomposition steps only.
"""
                    }],
                    temperature=0.2,
                    max_tokens=200
                )
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_ref_mgdebugger = evaluate_method(ref_mgdebugger_solve, "MGDebugger [6]", delay=25)
save_result("ref_mgdebugger", acc_ref_mgdebugger)


EVALUATING: MGDebugger [6]

[1/30] has_close_elements
PASSED (attempt 1)
PASS (1/1)

[2/30] separate_paren_groups
PASSED (attempt 1)
PASS (2/2)

[3/30] truncate_number
PASSED (attempt 1)
PASS (3/3)

[4/30] below_zero
PASSED (attempt 1)
PASS (4/4)

[5/30] mean_absolute_deviation
PASSED (attempt 1)
PASS (5/5)

[6/30] intersperse
PASSED (attempt 1)
PASS (6/6)

[7/30] parse_nested_parens
PASSED (attempt 1)
PASS (7/7)

[8/30] filter_by_substring
PASSED (attempt 1)
PASS (8/8)

[9/30] sum_product
PASSED (attempt 1)
PASS (9/9)

[10/30] rolling_max
PASSED (attempt 1)
PASS (10/10)

[11/30] make_palindrome
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (10/11)

[12/30] string_xor
PASSED (attempt 1)
PASS (11/12)

[13/30] longest
FAIL (attempt 1): name 'List' is not defined
PASSED (attempt 2)
PASS (12/13)

[14/30] greatest_common_divisor
PASSED (attempt 1)
PASS (13/14)

[15/30] all_prefixes
FAIL (attempt 1): name 'List' is not defined
PASSED (attempt 2)
PASS (14/15)

[16/30] string_

In [ ]:
with open(RESULTS_PATH, "r") as f:
    final_results = json.load(f)

reference_scores = {
    "standard_ma":       "-",
    "standard_sd":       "-",
    "standard_hybrid":   "-",
    "codesim_ma":        "95.1% [7]",
    "codesim_sd":        "90.7% [7]",
    "codesim_hybrid":    "-",
    "normal_agentcoder": "-",
    "normal_codex":      "-",
    "normal_mgdebugger": "-",
    "ref_agentcoder":    "96.3% [19]",
    "ref_codex":         "28.8% [9]",
    "ref_mgdebugger":    "94.5% [6]",
}

labels = {
    "standard_ma":       "Standard Multi-Agent",
    "standard_sd":       "Standard Self-Debug",
    "standard_hybrid":   "Standard Hybrid",
    "codesim_ma":        "CODESIM Multi-Agent",
    "codesim_sd":        "CODESIM Self-Debug",
    "codesim_hybrid":    "CODESIM Hybrid",
    "normal_agentcoder": "Normal AgentCoder",
    "normal_codex":      "Normal Codex Baseline",
    "normal_mgdebugger": "Normal MGDebugger",
    "ref_agentcoder":    "AgentCoder [19]",
    "ref_codex":         "Codex Baseline [9]",
    "ref_mgdebugger":    "MGDebugger [6]",
}

print("\n" + "=" * 70)
print("FINAL RESULTS - HUMANEVAL (n=30)")
print("=" * 70)
print(f"{'Method':<30} {'Our Result':>12} {'Reference':>15}")
print("-" * 60)
for key, label in labels.items():
    our = final_results.get(key, "N/A")
    ref = reference_scores.get(key, "-")
    if isinstance(our, float):
        print(f"{label:<30} {our:>10.1f}% {ref:>15}")
    else:
        print(f"{label:<30} {'N/A':>11} {ref:>15}")
print("=" * 70)


FINAL RESULTS - HUMANEVAL (n=30)
Method                           Our Result       Reference
------------------------------------------------------------
Standard Multi-Agent                 96.7%               -
Standard Self-Debug                 100.0%               -
Standard Hybrid                      96.7%               -
CODESIM Multi-Agent                  96.7%       95.1% [7]
CODESIM Self-Debug                   96.7%       90.7% [7]
CODESIM Hybrid                       96.7%               -
Normal AgentCoder                    96.7%               -
Normal Codex Baseline                86.7%               -
Normal MGDebugger                    96.7%               -
AgentCoder [19]                     100.0%      96.3% [19]
Codex Baseline [9]                   96.7%       28.8% [9]
MGDebugger [6]                       96.7%       94.5% [6]


In [ ]:
#   Codex Baseline n=50
def codex_baseline_50(problem, test_code):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a Python function to solve the following problem.

Problem:
{problem}

Return ONLY the Python function.
No explanation. No markdown.
"""
            }],
            temperature=0.0,
            max_tokens=400
        ))
        run_code(code, test_code)
        print("PASSED")
        return True
    except Exception as e:
        print(f"FAIL: {str(e)[:60]}")
        return False


passed = 0
results_50 = {}

for idx, sample in enumerate(dataset.select(range(50))):
    print(f"\n[{idx+1}/50] {sample['entry_point']}")
    result = codex_baseline_50(sample["prompt"], make_test_code(sample))
    if result:
        passed += 1
    print(f"Score: {passed}/{idx+1}")
    time.sleep(15)

acc_codex_50 = (passed / 50) * 100
results_50["codex_50"] = acc_codex_50
print(f"\nCodex Baseline n=50: {passed}/50 = {acc_codex_50:.1f}%")
save_result("codex_50", acc_codex_50)


[1/50] has_close_elements
PASSED
Score: 1/1

[2/50] separate_paren_groups
PASSED
Score: 2/2

[3/50] truncate_number
PASSED
Score: 3/3

[4/50] below_zero
PASSED
Score: 4/4

[5/50] mean_absolute_deviation
PASSED
Score: 5/5

[6/50] intersperse
PASSED
Score: 6/6

[7/50] parse_nested_parens
PASSED
Score: 7/7

[8/50] filter_by_substring
PASSED
Score: 8/8

[9/50] sum_product
PASSED
Score: 9/9

[10/50] rolling_max
PASSED
Score: 10/10

[11/50] make_palindrome
FAIL: name 'is_palindrome' is not defined
Score: 10/11

[12/50] string_xor
PASSED
Score: 11/12

[13/50] longest
FAIL: name 'Optional' is not defined
Score: 11/13

[14/50] greatest_common_divisor
PASSED
Score: 12/14

[15/50] all_prefixes
FAIL: name 'List' is not defined
Score: 12/15

[16/50] string_sequence
PASSED
Score: 13/16

[17/50] count_distinct_characters
PASSED
Score: 14/17

[18/50] parse_music
PASSED
Score: 15/18

[19/50] how_many_times
PASSED
Score: 16/19

[20/50] sort_numbers
PASSED
Score: 17/20

[21/50] find_closest_elements
PAS

In [ ]:
#  Standard Multi-Agent n=50
passed = 0

for idx, sample in enumerate(dataset.select(range(50))):
    print(f"\n[{idx+1}/50] {sample['entry_point']}")
    result = standard_ma_solve(sample["prompt"], make_test_code(sample))
    if result:
        passed += 1
    print(f"Score: {passed}/{idx+1}")
    time.sleep(20)

acc_standard_ma_50 = (passed / 50) * 100
results_50["standard_ma_50"] = acc_standard_ma_50
print(f"\nStandard Multi-Agent n=50: {passed}/50 = {acc_standard_ma_50:.1f}%")
save_result("standard_ma_50", acc_standard_ma_50)


[1/50] has_close_elements
PASSED (attempt 1)
Score: 1/1

[2/50] separate_paren_groups
PASSED (attempt 1)
Score: 2/2

[3/50] truncate_number
PASSED (attempt 1)
Score: 3/3

[4/50] below_zero
PASSED (attempt 1)
Score: 4/4

[5/50] mean_absolute_deviation
PASSED (attempt 1)
Score: 5/5

[6/50] intersperse
PASSED (attempt 1)
Score: 6/6

[7/50] parse_nested_parens
PASSED (attempt 1)
Score: 7/7

[8/50] filter_by_substring
PASSED (attempt 1)
Score: 8/8

[9/50] sum_product
PASSED (attempt 1)
Score: 9/9

[10/50] rolling_max
PASSED (attempt 1)
Score: 10/10

[11/50] make_palindrome
FAIL (attempt 1): name 'is_palindrome' is not defined
FAIL (attempt 2): 
FAIL (attempt 3): name 'is_palindrome' is not defined
Score: 10/11

[12/50] string_xor
PASSED (attempt 1)
Score: 11/12

[13/50] longest
PASSED (attempt 1)
Score: 12/13

[14/50] greatest_common_divisor
PASSED (attempt 1)
Score: 13/14

[15/50] all_prefixes
PASSED (attempt 1)
Score: 14/15

[16/50] string_sequence
PASSED (attempt 1)
Score: 15/16

[17/50

In [ ]:
#  CODESIM Multi-Agent n=50
passed = 0

for idx, sample in enumerate(dataset.select(range(50))):
    print(f"\n[{idx+1}/50] {sample['entry_point']}")
    result = codesim_ma_solve(sample["prompt"], make_test_code(sample))
    if result:
        passed += 1
    print(f"Score: {passed}/{idx+1}")
    time.sleep(20)

acc_codesim_ma_50 = (passed / 50) * 100
results_50["codesim_ma_50"] = acc_codesim_ma_50
print(f"\nCODESIM Multi-Agent n=50: {passed}/50 = {acc_codesim_ma_50:.1f}%")
save_result("codesim_ma_50", acc_codesim_ma_50)


[1/50] has_close_elements
PASSED (attempt 1)
Score: 1/1

[2/50] separate_paren_groups
PASSED (attempt 1)
Score: 2/2

[3/50] truncate_number
PASSED (attempt 1)
Score: 3/3

[4/50] below_zero
PASSED (attempt 1)
Score: 4/4

[5/50] mean_absolute_deviation
PASSED (attempt 1)
Score: 5/5

[6/50] intersperse
PASSED (attempt 1)
Score: 6/6

[7/50] parse_nested_parens
PASSED (attempt 1)
Score: 7/7

[8/50] filter_by_substring
PASSED (attempt 1)
Score: 8/8

[9/50] sum_product
PASSED (attempt 1)
Score: 9/9

[10/50] rolling_max
PASSED (attempt 1)
Score: 10/10

[11/50] make_palindrome
FAIL (attempt 1): 
FAIL (attempt 2): name 'is_palindrome' is not defined
FAIL (attempt 3): 
Score: 10/11

[12/50] string_xor
PASSED (attempt 1)
Score: 11/12

[13/50] longest
PASSED (attempt 1)
Score: 12/13

[14/50] greatest_common_divisor
PASSED (attempt 1)
Score: 13/14

[15/50] all_prefixes
PASSED (attempt 1)
Score: 14/15

[16/50] string_sequence
PASSED (attempt 1)
Score: 15/16

[17/50] count_distinct_characters
PASSED 

In [ ]:
print("\n" + "=" * 65)
print("EFFECT OF SAMPLE SIZE - HUMANEVAL")
print("=" * 65)
print(f"{'Method':<25} {'n=30':>10} {'n=50':>10} {'Difference':>12}")
print("-" * 55)

comparison = [
    ("Codex Baseline",      86.7, acc_codex_50),
    ("Standard Multi-Agent", 96.7, acc_standard_ma_50),
    ("CODESIM Multi-Agent",  96.7, acc_codesim_ma_50),
]

for method, n30, n50 in comparison:
    diff = n50 - n30
    sign = "+" if diff >= 0 else ""
    print(f"{method:<25} {n30:>9.1f}% {n50:>9.1f}% {sign}{diff:>10.1f}%")

print("=" * 65)
print("\nOBSERVATIONS:")
print("- Larger sample size reveals more realistic accuracy differences")
print("- Ceiling effect is reduced with n=50 vs n=30")
print("- CODESIM advantage becomes clearer with more samples")


EFFECT OF SAMPLE SIZE - HUMANEVAL
Method                          n=30       n=50   Difference
-------------------------------------------------------
Codex Baseline                 86.7%      84.0%       -2.7%
Standard Multi-Agent           96.7%      94.0%       -2.7%
CODESIM Multi-Agent            96.7%      98.0% +       1.3%

OBSERVATIONS:
- Larger sample size reveals more realistic accuracy differences
- Ceiling effect is reduced with n=50 vs n=30
- CODESIM advantage becomes clearer with more samples
